In [ ]:
!pip install langchain langchain-community langchain-groq chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/7

In [ ]:
from google.colab import files

uploaded = files.upload()          # click and choose a PDF

pdf_name = list(uploaded.keys())[0]

print('Uploaded:', pdf_name)

Saving IOT Lab2 .pdf to IOT Lab2 .pdf
Uploaded: IOT Lab2 .pdf


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.0 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter


pages = PyPDFLoader(pdf_name).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = splitter.split_documents(pages)

print('Number of chunks:', len(chunks))

/tmp/ipykernel_1575/1376468322.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of chunks: 20


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize a free, high-quality embeddings model (No API key needed!)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector database and name it 'db'
db = Chroma.from_documents(chunks, embeddings)
print("Vector database created successfully!")

/tmp/ipykernel_1575/1358679043.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully!


In [ ]:
!pip install langchain-classic

In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize your LLM
llm = ChatGroq(
    api_key="YOUR_GROQ_API_KEY",
    model="llama-3.1-8b-instant",
    temperature=0
)

# 2. Define a simple prompt for the retrieval chain
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Create the chains
question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa = create_retrieval_chain(db.as_retriever(), question_answer_chain)

# 4. Run the query
query = 'What is this document about?'
response = qa.invoke({"input": query})
print(response['answer'])

This document appears to be a laboratory report (LAB Report) submitted by Vidya PB, a student pursuing a B.Tech in Computer Science. The report seems to be related to Internet of Things (IoT) projects, specifically focusing on different aspects of IoT communication.

The report covers three main topics:

1. UART Serial Communication
2. Wi-Fi Connectivity using ESP32
3. Secure IoT Communication (Basic Encryption / Auth)

Each section includes a hardware connection diagram and sample code, along with expected outputs. This suggests that the report is a hands-on exploration of various IoT communication techniques, likely as part of a course assignment or project.


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key="YOUR_GROQ_API_KEY",

               model="llama-3.1-8b-instant", temperature=0)

In [ ]:
def answer_from_pdf(query):

    docs = db.as_retriever().invoke(query)

    context = '\n'.join(d.page_content for d in docs)

    prompt = (f"Use ONLY this context to answer.\n{context}\n\n"

              f"Question: {query}\n"

              "If the context does not contain the answer, reply exactly: "

              "'I don't know'.")

    return llm.invoke(prompt).content

In [ ]:
def adaptive_answer(question, max_tries=3):
    query = question
    for attempt in range(1, max_tries + 1):
        print(f'Attempt {attempt} with query: {query}')
        answer = answer_from_pdf(query)
        if "i don't know" not in answer.lower():
            return answer          # good answer, stop
        # otherwise, rephrase and retry
        query = llm.invoke(
            f'Rephrase this search query differently: {query}'
        ).content
    return 'Could not find an answer after several tries.'

# This line must have ZERO indentation (start at the very beginning of the line)
print(adaptive_answer('What is the main conclusion of the document?'))

Attempt 1 with query: What is the main conclusion of the document?
Attempt 2 with query: Here are a few alternative search queries:

1. What is the key takeaway from the document?
2. What is the author's main point or finding?
3. What is the summary or conclusion of the document?
4. What is the central argument or thesis of the document?
5. What is the final verdict or recommendation of the document?

These rephrased search queries can help you refine your search and get more accurate results.
Attempt 3 with query: Here are some rephrased search queries based on the given alternatives:

1. **Document overview**: This search query can help you find a concise summary of the document's main points.
2. **Author's stance**: This query can provide information on the author's perspective or opinion on the topic.
3. **Document summary**: This search query can yield a brief summary of the document's key points and findings.
4. **Document thesis statement**: This query can help you identify the 